# Module 05-1: 프롬프트 구조 설계 (솔루션)

## 학습 목표
- System/Human 메시지를 분리하여 `ChatPromptTemplate.from_messages()`를 구성할 수 있다
- `format_messages()`로 변수를 치환하고 결과를 확인할 수 있다
- 프롬프트 4요소(역할/맥락/지시/형식)로 코드 리뷰 프롬프트를 설계할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/05-prompt-engineering.md` 섹션 1.2, 1.3, 2.1

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: ChatPromptTemplate 기본 구성 (솔루션)

In [ ]:
# 솔루션
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 코드 리뷰 전문가입니다.\n주어진 코드의 버그와 개선점을 분석하세요.\n반드시 JSON 형식으로 응답하세요."),
    ("human", "## 분석할 코드\n언어: {language}\n```\n{code}\n```\n\n위 코드를 분석해주세요."),
])

formatted = prompt.format_messages(language="Python", code="def divide(a, b):\n    return a / b")

for msg in formatted:
    print(f"[{msg.type}]")
    print(msg.content[:200])
    print("-" * 40)

In [ ]:
# 검증 셀
assert prompt is not None
assert formatted is not None
assert len(formatted) == 2
assert formatted[0].type == "system"
assert formatted[1].type == "human"
assert "Python" in formatted[1].content
assert "divide" in formatted[1].content
print("실습 1 완료!")

## 실습 2: 체인 파이프라인 구성 (솔루션)

In [ ]:
# 솔루션
llm = FakeLLM(responses={
    "divide": '{"bugs": [{"line": 2, "issue": "ZeroDivisionError when b=0", "severity": "HIGH"}], "score": 5}'
})
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"language": "Python", "code": "def divide(a, b):\n    return a / b"})

print("체인 실행 결과:")
print(result)

In [ ]:
# 검증 셀
assert llm is not None
assert chain is not None
assert result is not None
assert isinstance(result, str)
print("실습 2 완료!")

## 실습 3: 코드 리뷰 프롬프트를 4요소로 설계 (솔루션)

In [ ]:
# 솔루션
code_review_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "[역할] 당신은 시니어 Python 코드 리뷰어입니다.\n\n"
     "[맥락] 이 프로젝트는 Python/FastAPI 기반 REST API입니다.\n\n"
     "[지시] 아래 코드의 버그, 보안 이슈, 성능 개선점을 분석하세요.\n\n"
     '[형식] JSON으로 응답하세요: {"bugs": [...], "security": [...], "score": 1-10}'
    ),
    ("human", "파일: {file_path}\n\n```python\n{code}\n```"),
])

review_llm = FakeLLM(responses={
    "authenticate": '{"bugs": [{"line": 2, "issue": "SQL Injection", "severity": "HIGH"}], "security": [{"issue": "하드코딩 패스워드"}], "score": 2}'
})

review_chain = code_review_prompt | review_llm | StrOutputParser()
review_result = review_chain.invoke({
    "file_path": "auth.py",
    "code": "def authenticate(pw):\n    if pw == 'admin': return True"
})

print("코드 리뷰 결과:")
print(review_result)

In [ ]:
# 검증 셀
assert code_review_prompt is not None
assert review_chain is not None
assert review_result is not None

system_content = code_review_prompt.messages[0].prompt.template
keywords_found = [kw for kw in ["역할", "맥락", "지시", "형식"] if kw in system_content]
assert len(keywords_found) >= 3, f"4요소가 충분히 포함되지 않았습니다. 발견된 요소: {keywords_found}"

test_formatted = code_review_prompt.format_messages(file_path="test.py", code="pass")
assert "test.py" in test_formatted[1].content
print("실습 3 완료!")

## 정리

1. **ChatPromptTemplate.from_messages()**: System/Human 메시지를 분리하여 구조화된 프롬프트 생성
2. **변수 치환**: `{variable}` 플레이스홀더를 `format_messages()` 또는 `invoke()`로 치환
3. **체인 파이프라인**: `prompt | llm | parser` 패턴으로 데이터 흐름 구성
4. **4요소 설계**: 역할/맥락/지시/형식을 System 메시지에 구조적으로 배치